In [1]:
from pepbenchmark.pep_utils import convert

/home/batchcom/assist/miniforge3/envs/pepbenchmark/lib/python3.10/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [ ]:
import os
import numpy as np
import pandas as pd

from pepbenchmark.pep_utils import convert

project_root = "/home/batchcom/assist/pepbenchmark/final/pepbenchmark"
data_root = os.path.join(project_root, "PepBenchData")
# model_name = "facebook/esm2_t30_150M_UR50D"
model_name = "airkingbd/dplm_150m"
fasta_paths = []
for root, _, files in os.walk(data_root):
    # if "fasta.csv" in files:
    #     fasta_paths.append(os.path.join(root, "fasta.csv"))
    # if "prot_fasta.csv" in files:
    #     fasta_paths.append(os.path.join(root, "prot_fasta.csv"))
    if "pep_fasta.csv" in files:
        fasta_paths.append(os.path.join(root, "pep_fasta.csv"))

fasta_paths.sort()# print(f"Found {len(fasta_paths)} fasta.csv files")
print(f"Found {len(fasta_paths)} pep_fasta.csv files")
if not fasta_paths:
    raise SystemExit(0)

embedder = convert.Fasta2Embedding(model_name)

for i, fasta_path in enumerate(fasta_paths, 1):
    # out_path = os.path.join(os.path.dirname(fasta_path), "esm2_150_embedding.npz")
    out_path = os.path.join(os.path.dirname(fasta_path), "prot_esm2_150_embedding.npz")
    # out_path = os.path.join(os.path.dirname(fasta_path), "prot_dplm_150_embedding.npz")
    # out_path = os.path.join(os.path.dirname(fasta_path), "pep_dplm_150_embedding.npz")
    df = pd.read_csv(fasta_path)
    if "feature" not in df.columns:
        print(f"[{i}/{len(fasta_paths)}] SKIP (missing 'feature' column): {fasta_path}")
        continue

    seqs = df["feature"].astype(str).tolist()
    if len(seqs) == 0:
        print(f"[{i}/{len(fasta_paths)}] SKIP (empty fasta): {fasta_path}")
        continue

    emb = embedder(seqs, batch_size=512)
    np.savez_compressed(out_path, data=emb)
    print(f"[{i}/{len(fasta_paths)}] OK {out_path} shape={emb.shape}")

print("All done")


Found 2 pep_fasta.csv files


Some weights of EsmModel were not initialized from the model checkpoint at airkingbd/dplm_150m and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[1/2] OK /home/batchcom/assist/pepbenchmark/final/pepbenchmark/PepBenchData/PepBenchData-50/PpI/pep_dplm_150_embedding.npz shape=(44148, 640)
[2/2] OK /home/batchcom/assist/pepbenchmark/final/pepbenchmark/PepBenchData/PepBenchData-50/PpI_ba/pep_dplm_150_embedding.npz shape=(1433, 640)
All done


In [ ]:
from pepbenchmark.pep_utils import convert
# available transforms
print(convert.FormatTransform.supported_transforms())
fasta2emb =  convert.Fasta2Embedding("facebook/esm2_t30_150M_UR50D")
embedding = fasta2emb(["ALAGGGPCR","ALLLG"])
print(embedding.shape)  # (640,) for ESM-2 150M model

helm2fasta =  convert.Helm2Fasta()
helm2smiles = convert.Helm2Smiles()
helm ="PEPTIDE1{[d(N->O)Gly(allyl)].P.I.[meV].[meA].[bAla]}$PEPTIDE1,PEPTIDE1,1:R1-6:R2$$V2.0"

print(helm2fasta(helm))
print(helm2smiles(helm))

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t30_150M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


(2, 640)
